<a href="https://colab.research.google.com/github/UmerSajid842/Fraud-detection-system/blob/main/grokieeecode.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Recommendation: Design a customized hybrid model aligned with your title rather than relying only on an off-the-shelf model.
The IEEE-CIS Fraud Detection dataset (≈590k transactions, ~3.5% fraud rate, highly imbalanced, TransactionDT as time delta, card/addr/email/device features, many Vesta-engineered V features) has strong relational structure (shared cards, devices, emails, addresses) and temporal dynamics. Pure tabular models miss multi-hop fraud rings and time-evolving patterns; pure sequence models miss entity graphs. A custom hybrid that fuses Adaptive Transformers (for sequential/temporal patterns), Graph Proposal Neural Networks / Graph Transformers (for relational structure), and an LLM explanation head matches the title and the data characteristics best.
3–4 Strong Candidate Models

Custom Adaptive Spatio-Temporal Graph Transformer (recommended primary)Adaptive Transformer + Graph Proposal / Graph Transformer + LLM explanation head.Best match to your title and the dataset.
Temporal Graph Network (TGN) or Temporal Graph Attention NetworkStrong baseline for continuous-time dynamic graphs of transactions.
GraphSAGE / GAT / FraudGT-style Graph TransformerExcellent for static or snapshot graphs of cards–devices–transactions; can be extended with temporal encodings.
TabTransformer / FT-Transformer or pure Transformer on engineered featuresStrong tabular baseline, easier to implement, but weaker on explicit multi-entity relations and long-range temporal dependencies.

Why the custom Adaptive Spatio-Temporal Graph Transformer is mathematically and technically superior for this task

Class imbalance & rare fraud patterns: Standard cross-entropy is dominated by the majority class. Focal loss or class-balanced loss focuses gradient on hard minority examples:$$FL(p_t) = -\alpha_t (1-p_t)^\gamma \log(p_t)$$Combined with graph message passing, rare fraudulent subgraphs receive amplified signal.
Relational (spatial) structure: Fraud often propagates through shared entities (same card, device, email, IP). Graph Neural Networks / Graph Transformers perform neighborhood aggregation:$$\mathbf{h}_v^{(l+1)} = \sigma\Bigl(\mathbf{W}\cdot\mathrm{AGG}\bigl(\{\mathbf{h}_u^{(l)}:u\in\mathcal{N}(v)\}\bigr)\Bigr)$$or multi-head attention over neighbors. This captures multi-hop fraud rings that tabular models cannot see.
Temporal dynamics: TransactionDT defines a continuous time axis. Adaptive / temporal attention or continuous-time kernels (as in TGN) weight recent vs. historical events:$$\alpha_{ij} = \mathrm{softmax}\Bigl(\frac{(\mathbf{Q}\mathbf{h}_i)^\top\mathbf{K}\mathbf{h}_j}{\sqrt{d}} + f(\Delta t_{ij})\Bigr)$$where $f(\Delta t)$ is a learned time encoding or decay. This models evolving fraud tactics.
Heterogeneous features + missing identity data: Adaptive attention or feature-token Transformers handle mixed categorical/numerical features and missing values better than fixed embeddings. Graph proposal modules can generate candidate edges from card/device/email co-occurrence.
Explainability: An LLM head conditioned on the attention weights / subgraph produces human-readable, trustworthy rationales (required by your title).

Existing pure models (XGBoost, vanilla Transformer, plain GCN) are useful baselines but lack the joint spatio-temporal inductive bias needed for state-of-the-art results on this data.

Modular Pipeline Design
The pipeline is organized so each module’s output is the next module’s input and is persisted to CSV/Excel for inspection and reproducibility.
text
Raw Data
  → Module 1: Data Loading & Merging
  → Module 2: Preprocessing & Feature Engineering
  → Module 3: Graph Construction (nodes/edges)
  → Module 4: Adaptive Spatio-Temporal Model (Transformer + Graph)
  → Module 5: Training Loop
  → Module 6: Evaluation (PR-AUC, ROC-AUC, Precision, Recall, F1, MCC)
  → Module 7: LLM-based Trustworthy Explanations (optional post-hoc)

In [1]:
#Prerequisites (install once):
#Python

!pip install pandas numpy scikit-learn torch torch-geometric networkx openpyxl matplotlib seaborn

# later: transformers openai / huggingface

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 15.5 MB/s eta 0:00:00


Module 1: Data Loading & Merging
Purpose: Load the two IEEE-CIS tables, join on TransactionID, keep only useful columns, and persist the merged table.
Python

In [2]:
# ============================================================
# MODULE 1: DATA LOADING AND MERGING
# Purpose: Load transaction + identity tables, left-join on TransactionID,
#          select core columns, and save the merged dataframe.
# ============================================================

import pandas as pd
import numpy as np
import os

# Create output directory for all intermediate artifacts
os.makedirs("artifacts", exist_ok=True)

def load_and_merge_data(transaction_path: str, identity_path: str) -> pd.DataFrame:
    """
    Load the two CSV files and perform a left join.
    Not every transaction has identity information → left join preserves all transactions.
    """
    # Read transaction table (main table with isFraud and most features)
    df_trans = pd.read_csv(transaction_path)
    # Read identity table (device, browser, IP-related features)
    df_id = pd.read_csv(identity_path)

    # Left join keeps every transaction; identity columns become NaN when missing
    df = df_trans.merge(df_id, on="TransactionID", how="left")

    # Optional: keep only the most informative columns to reduce memory
    # (you can expand this list later)
    keep_cols = [
        "TransactionID", "isFraud", "TransactionDT", "TransactionAmt",
        "ProductCD", "card1", "card2", "card3", "card4", "card5", "card6",
        "addr1", "addr2", "dist1", "dist2",
        "P_emaildomain", "R_emaildomain",
        "C1", "C2", "C3", "C4", "C5", "C6", "C7", "C8", "C9", "C10",
        "C11", "C12", "C13", "C14",
        "D1", "D2", "D3", "D4", "D5", "D10", "D15",
        "M1", "M2", "M3", "M4", "M5", "M6", "M7", "M8", "M9",
        "DeviceType", "DeviceInfo",
        "id_01", "id_02", "id_05", "id_06", "id_11", "id_12", "id_13",
        "id_15", "id_16", "id_17", "id_19", "id_20", "id_28", "id_29",
        "id_30", "id_31", "id_35", "id_36", "id_37", "id_38"
    ]
    # Keep only columns that actually exist in the merged frame
    keep_cols = [c for c in keep_cols if c in df.columns]
    df = df[keep_cols]

    return df

# ---------- Execution ----------
# Replace paths with your actual file locations
# df_raw = load_and_merge_data("train_transaction.csv", "train_identity.csv")

# For demonstration / testing without the full 1.3 GB data you can create a tiny synthetic sample:
np.random.seed(42)
n = 5000
df_raw = pd.DataFrame({
    "TransactionID": range(n),
    "isFraud": np.random.binomial(1, 0.035, n),
    "TransactionDT": np.sort(np.random.randint(0, 1_000_000, n)),
    "TransactionAmt": np.random.lognormal(3, 1, n),
    "ProductCD": np.random.choice(["W", "H", "C", "S", "R"], n),
    "card1": np.random.randint(1000, 20000, n),
    "card4": np.random.choice(["visa", "mastercard", "discover", "american express"], n),
    "card6": np.random.choice(["debit", "credit"], n),
    "addr1": np.random.randint(100, 500, n),
    "P_emaildomain": np.random.choice(["gmail.com", "yahoo.com", "hotmail.com", None], n),
    "DeviceType": np.random.choice(["desktop", "mobile", None], n),
    "C1": np.random.poisson(5, n),
    "D1": np.random.exponential(30, n),
})

# Persist Module 1 output
df_raw.to_csv("artifacts/01_merged_raw.csv", index=False)
print("Module 1 complete. Shape:", df_raw.shape)
print("Fraud rate:", df_raw["isFraud"].mean())

Module 1 complete. Shape: (5000, 13)
Fraud rate: 0.035


Module 2: Preprocessing & Feature Engineering
Purpose: Handle missing values, encode categoricals, create time features, scale numerics, and produce a clean feature matrix + labels. Output is saved for the next stage.

In [3]:
# ============================================================
# MODULE 2: PREPROCESSING & FEATURE ENGINEERING
# Purpose: Impute missing values, encode categoricals, engineer
#          temporal features, scale numerics, and save clean matrix.
# ============================================================

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

def preprocess_and_engineer(df: pd.DataFrame):
    """
    Full preprocessing pipeline.
    Returns:
        X          – feature matrix (DataFrame)
        y          – binary labels
        feature_names – list of column names used
        cat_cols   – list of categorical column names (for later embedding)
        num_cols   – list of numeric column names
    """
    df = df.copy()

    # ---------- 2.1 Temporal features from TransactionDT ----------
    # TransactionDT is a timedelta (seconds from an arbitrary origin)
    df["hour"] = (df["TransactionDT"] // 3600) % 24          # hour of day
    df["day"]  = (df["TransactionDT"] // 86400) % 7          # day of week proxy
    df["log_amt"] = np.log1p(df["TransactionAmt"])           # log-transform amount

    # ---------- 2.2 Missing-value handling ----------
    # Numeric columns → median imputation
    num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    num_cols = [c for c in num_cols if c not in ["TransactionID", "isFraud"]]
    for c in num_cols:
        df[c] = df[c].fillna(df[c].median())

    # Categorical columns → "missing" token
    cat_cols = df.select_dtypes(include=["object"]).columns.tolist()
    for c in cat_cols:
        df[c] = df[c].fillna("missing").astype(str)

    # ---------- 2.3 Label encoding for categoricals ----------
    # We use LabelEncoder for simplicity; for production consider target encoding
    # or embedding layers inside the model.
    label_encoders = {}
    for c in cat_cols:
        le = LabelEncoder()
        df[c] = le.fit_transform(df[c])
        label_encoders[c] = le

    # ---------- 2.4 Final feature matrix ----------
    feature_cols = [c for c in df.columns if c not in ["TransactionID", "isFraud"]]
    X = df[feature_cols]
    y = df["isFraud"].astype(int)

    # Optional: scale numeric features (Transformers often work better with scaled inputs)
    scaler = StandardScaler()
    X[num_cols] = scaler.fit_transform(X[num_cols])

    return X, y, feature_cols, cat_cols, num_cols, label_encoders, scaler

# ---------- Execution ----------
X, y, feature_names, cat_cols, num_cols, label_encoders, scaler = preprocess_and_engineer(df_raw)

# Persist Module 2 outputs
X.to_csv("artifacts/02_features.csv", index=False)
pd.Series(y, name="isFraud").to_csv("artifacts/02_labels.csv", index=False)
pd.DataFrame({"feature": feature_names}).to_csv("artifacts/02_feature_names.csv", index=False)

print("Module 2 complete. Feature matrix shape:", X.shape)
print("Categorical columns:", cat_cols)
print("Numeric columns sample:", num_cols[:5])

Module 2 complete. Feature matrix shape: (5000, 14)
Categorical columns: ['ProductCD', 'card4', 'card6', 'P_emaildomain', 'DeviceType']
Numeric columns sample: ['TransactionDT', 'TransactionAmt', 'card1', 'addr1', 'C1']


/tmp/ipykernel_3297/2634393169.py:56: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X[num_cols] = scaler.fit_transform(X[num_cols])


Module 3: Graph Construction
Purpose: Build a transaction graph where nodes are transactions (or cards/devices) and edges encode shared identifiers. Save edge list and node features for the GNN/Transformer.

In [4]:
# ============================================================
# MODULE 3: GRAPH CONSTRUCTION
# Purpose: Create a graph from shared entities (card, device, email…).
#          Nodes = transactions; edges = shared card1 / DeviceType / email.
#          Save edge_index and node features for PyTorch Geometric.
# ============================================================

import torch
from torch_geometric.data import Data
from collections import defaultdict

def build_transaction_graph(df: pd.DataFrame, X: pd.DataFrame, y: pd.Series,
                            entity_cols=("card1", "P_emaildomain", "DeviceType")):
    """
    Build an undirected multi-relational graph.
    For each entity column we connect all transactions that share the same value.
    To keep the graph tractable we sample a limited number of edges per entity.
    """
    n = len(df)
    edge_src, edge_dst = [], []

    for col in entity_cols:
        if col not in df.columns:
            continue
        # Group transaction indices by entity value
        groups = df.groupby(col).groups
        for val, idxs in groups.items():
            idxs = list(idxs)
            if len(idxs) < 2:
                continue
            # Connect consecutive transactions in time order (simple chain)
            # or create a star / fully-connected clique (more expensive)
            idxs_sorted = sorted(idxs, key=lambda i: df.loc[i, "TransactionDT"])
            for i in range(len(idxs_sorted) - 1):
                a, b = idxs_sorted[i], idxs_sorted[i + 1]
                edge_src.append(a)
                edge_dst.append(b)
                edge_src.append(b)          # undirected
                edge_dst.append(a)

    # Convert to tensors
    edge_index = torch.tensor([edge_src, edge_dst], dtype=torch.long)

    # Node features = the preprocessed feature matrix
    x = torch.tensor(X.values, dtype=torch.float)
    y_tensor = torch.tensor(y.values, dtype=torch.long)

    data = Data(x=x, edge_index=edge_index, y=y_tensor)

    # Also save a simple edge list for inspection
    edge_df = pd.DataFrame({"src": edge_src, "dst": edge_dst})
    edge_df.to_csv("artifacts/03_edge_list.csv", index=False)

    return data

# ---------- Execution ----------
# Reconstruct a minimal df that still has the entity columns used for edges
df_for_graph = df_raw.copy()
# Make sure the index of X matches df_raw
graph_data = build_transaction_graph(df_for_graph, X, y)

print("Module 3 complete.")
print("Number of nodes:", graph_data.num_nodes)
print("Number of edges:", graph_data.num_edges)
torch.save(graph_data, "artifacts/03_graph_data.pt")

Module 3 complete.
Number of nodes: 5000
Number of edges: 15314


Module 4: Adaptive Spatio-Temporal Model Definition
Purpose: Define the hybrid architecture – Graph Attention / Transformer layers + temporal encoding + final classification head.
Python

In [5]:
# ============================================================
# MODULE 4: MODEL DEFINITION
# Purpose: Adaptive Spatio-Temporal Graph Transformer.
#          - Temporal positional encoding from TransactionDT
#          - Multi-head Graph Attention (spatial)
#          - Transformer-style self-attention on node embeddings
#          - Binary classification head
# ============================================================

import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GATConv, global_mean_pool

class AdaptiveSpatioTemporalFraudModel(nn.Module):
    """
    Hybrid model:
    1. Linear projection of raw features
    2. Temporal encoding (sinusoidal or learned) added to node features
    3. Stack of GAT layers (spatial message passing)
    4. Optional Transformer encoder on the resulting node embeddings
    5. Classification head
    """
    def __init__(self, in_dim: int, hidden_dim: int = 64, n_heads: int = 4,
                 n_gat_layers: int = 2, dropout: float = 0.2):
        super().__init__()
        self.input_proj = nn.Linear(in_dim, hidden_dim)

        # Simple learned temporal encoding (can be replaced by sinusoidal)
        self.time_emb = nn.Linear(1, hidden_dim)

        # Graph Attention layers
        self.gat_layers = nn.ModuleList()
        self.gat_layers.append(GATConv(hidden_dim, hidden_dim // n_heads,
                                       heads=n_heads, dropout=dropout))
        for _ in range(n_gat_layers - 1):
            self.gat_layers.append(GATConv(hidden_dim, hidden_dim // n_heads,
                                           heads=n_heads, dropout=dropout))

        # Lightweight Transformer encoder on node embeddings
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim, nhead=n_heads, dim_feedforward=hidden_dim * 2,
            dropout=dropout, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=1)

        # Classification head
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 1)          # binary logit
        )

        self.dropout = nn.Dropout(dropout)

    def forward(self, x, edge_index, time_feat=None, batch=None):
        """
        x          : [N, in_dim] node features
        edge_index : [2, E]
        time_feat  : [N, 1] normalized TransactionDT (optional)
        batch      : batch assignment for mini-batching (optional)
        """
        h = self.input_proj(x)                     # [N, hidden]

        # Add temporal information if provided
        if time_feat is not None:
            t = self.time_emb(time_feat)
            h = h + t

        # Spatial message passing
        for gat in self.gat_layers:
            h = gat(h, edge_index)
            h = F.elu(h)
            h = self.dropout(h)

        # Self-attention over nodes (treat the whole graph as a sequence)
        # For large graphs you would sample subgraphs or use linear attention
        h = h.unsqueeze(0)                         # [1, N, hidden]
        h = self.transformer(h)
        h = h.squeeze(0)                           # [N, hidden]

        # Node-level prediction
        logits = self.classifier(h).squeeze(-1)    # [N]
        return logits

# Instantiation example (will be used in training)
model = AdaptiveSpatioTemporalFraudModel(in_dim=X.shape[1])

Module 5: Training Loop
Purpose: Train the model with class-balanced / focal loss, early stopping, and checkpointing. Save predictions and loss history

In [6]:
# ============================================================
# MODULE 5: TRAINING LOOP
# Purpose: Train the hybrid model, handle class imbalance,
#          save best checkpoint and training history.
# ============================================================

from sklearn.model_selection import train_test_split
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau

def train_model(graph_data, X, y, epochs=30, lr=1e-3, hidden_dim=64):
    """
    Simple full-graph training (for illustration).
    For real IEEE-CIS scale you must use mini-batch neighbor sampling
    (torch_geometric.loader.NeighborLoader).
    """
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Train / validation split (stratified, time-aware split is better in practice)
    idx = np.arange(len(y))
    train_idx, val_idx = train_test_split(idx, test_size=0.2,
                                          stratify=y, random_state=42)

    model = AdaptiveSpatioTemporalFraudModel(
        in_dim=X.shape[1], hidden_dim=hidden_dim
    ).to(device)

    # Class weight to fight imbalance (fraud ≈ 3.5 %)
    pos_weight = torch.tensor([(y == 0).sum() / (y == 1).sum()], device=device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    optimizer = Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    scheduler = ReduceLROnPlateau(optimizer, mode="max", patience=3, factor=0.5)

    x = graph_data.x.to(device)
    edge_index = graph_data.edge_index.to(device)
    y_tensor = graph_data.y.float().to(device)

    # Optional time feature
    time_feat = torch.tensor(
        X["TransactionDT"].values if "TransactionDT" in X.columns
        else np.zeros(len(X)),
        dtype=torch.float
    ).unsqueeze(1).to(device)
    # Normalize time
    time_feat = (time_feat - time_feat.mean()) / (time_feat.std() + 1e-8)

    history = {"train_loss": [], "val_auc": []}
    best_val_auc = 0.0

    for epoch in range(1, epochs + 1):
        model.train()
        optimizer.zero_grad()
        logits = model(x, edge_index, time_feat)
        loss = criterion(logits[train_idx], y_tensor[train_idx])
        loss.backward()
        optimizer.step()

        # Validation
        model.eval()
        with torch.no_grad():
            val_logits = model(x, edge_index, time_feat)
            val_probs = torch.sigmoid(val_logits[val_idx]).cpu().numpy()
            val_true = y_tensor[val_idx].cpu().numpy()
            from sklearn.metrics import roc_auc_score
            val_auc = roc_auc_score(val_true, val_probs)

        scheduler.step(val_auc)
        history["train_loss"].append(loss.item())
        history["val_auc"].append(val_auc)

        if val_auc > best_val_auc:
            best_val_auc = val_auc
            torch.save(model.state_dict(), "artifacts/05_best_model.pt")

        if epoch % 5 == 0:
            print(f"Epoch {epoch:03d} | Loss {loss.item():.4f} | Val ROC-AUC {val_auc:.4f}")

    # Save training history
    pd.DataFrame(history).to_csv("artifacts/05_training_history.csv", index=False)
    print("Best validation ROC-AUC:", best_val_auc)
    return model, train_idx, val_idx

# ---------- Execution ----------
model, train_idx, val_idx = train_model(graph_data, X, y, epochs=20)

Epoch 005 | Loss 1.3463 | Val ROC-AUC 0.5397
Epoch 010 | Loss 1.3392 | Val ROC-AUC 0.5373
Epoch 015 | Loss 1.3307 | Val ROC-AUC 0.5299
Epoch 020 | Loss 1.3312 | Val ROC-AUC 0.5273
Best validation ROC-AUC: 0.5397039230199853


Module 6: Complete Fraud Evaluation Code
Purpose: Compute the full suite of metrics required for fraud detection (PR-AUC is especially important under heavy imbalance).

In [7]:
# ============================================================
# MODULE 6 (FIXED) – COMPLETE EVALUATION
# Run this AFTER the training cell has finished successfully.
# ============================================================
# ============================================================
# DIAGNOSTIC + IMPROVED TRAINING (RUN THIS NOW)
# ============================================================

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.metrics import average_precision_score, roc_auc_score
import numpy as np
import pandas as pd

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# ----------------------------------------------------------
# 1. Quick diagnostics
# ----------------------------------------------------------
print("\n===== DATA DIAGNOSTICS =====")
print("Total samples     :", len(y))
print("Fraud rate        :", y.mean().round(4))
print("Train fraud rate  :", y.values[train_idx].mean().round(4))
print("Val fraud rate    :", y.values[val_idx].mean().round(4))
print("Number of edges   :", graph_data.num_edges)
print("Feature dim       :", X.shape[1])

# Check if the model is collapsing to all-negative predictions
with torch.no_grad():
    model.eval()
    x = graph_data.x.to(device)
    edge_index = graph_data.edge_index.to(device)
    time_feat = torch.tensor(X["TransactionDT"].values if "TransactionDT" in X.columns
                             else np.zeros(len(X)), dtype=torch.float).unsqueeze(1).to(device)
    time_feat = (time_feat - time_feat.mean()) / (time_feat.std() + 1e-8)
    logits = model(x, edge_index, time_feat)
    probs = torch.sigmoid(logits).cpu().numpy()

print("\nPredicted probability statistics:")
print("  Min  :", probs.min().round(4))
print("  Max  :", probs.max().round(4))
print("  Mean :", probs.mean().round(4))
print("  >0.5 :", (probs > 0.5).sum())
print("  >0.3 :", (probs > 0.3).sum())
print("  >0.1 :", (probs > 0.1).sum())

# ----------------------------------------------------------
# 2. Improved model (slightly larger + residual)
# ----------------------------------------------------------
class ImprovedFraudModel(nn.Module):
    def __init__(self, in_dim, hidden_dim=128, n_heads=4, dropout=0.3):
        super().__init__()
        self.input_proj = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )
        self.time_emb = nn.Linear(1, hidden_dim)

        from torch_geometric.nn import GATConv
        self.gat1 = GATConv(hidden_dim, hidden_dim // n_heads, heads=n_heads, dropout=dropout)
        self.gat2 = GATConv(hidden_dim, hidden_dim // n_heads, heads=n_heads, dropout=dropout)

        self.norm1 = nn.LayerNorm(hidden_dim)
        self.norm2 = nn.LayerNorm(hidden_dim)

        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 1)
        )

    def forward(self, x, edge_index, time_feat=None):
        h = self.input_proj(x)
        if time_feat is not None:
            h = h + self.time_emb(time_feat)

        h = h + F.elu(self.gat1(h, edge_index))   # residual
        h = self.norm1(h)
        h = h + F.elu(self.gat2(h, edge_index))
        h = self.norm2(h)

        return self.classifier(h).squeeze(-1)

# ----------------------------------------------------------
# 3. Focal Loss (much better for extreme imbalance)
# ----------------------------------------------------------
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits, targets):
        bce = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        pt = torch.exp(-bce)
        focal = self.alpha * (1 - pt) ** self.gamma * bce
        return focal.mean()

# ----------------------------------------------------------
# 4. Re-train with improvements
# ----------------------------------------------------------
model = ImprovedFraudModel(in_dim=X.shape[1], hidden_dim=128).to(device)

# Stronger positive weight + Focal Loss
pos_weight = torch.tensor([(y == 0).sum() / max((y == 1).sum(), 1)], device=device)
criterion = FocalLoss(alpha=0.75, gamma=2.0)   # focus hard on frauds

optimizer = AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
scheduler = CosineAnnealingLR(optimizer, T_max=40)

x = graph_data.x.to(device)
edge_index = graph_data.edge_index.to(device)
y_tensor = graph_data.y.float().to(device)

time_feat = torch.tensor(
    X["TransactionDT"].values if "TransactionDT" in X.columns else np.zeros(len(X)),
    dtype=torch.float
).unsqueeze(1).to(device)
time_feat = (time_feat - time_feat.mean()) / (time_feat.std() + 1e-8)

print("\n===== STARTING IMPROVED TRAINING =====")
best_pr_auc = 0.0
history = []

for epoch in range(1, 41):
    model.train()
    optimizer.zero_grad()

    logits = model(x, edge_index, time_feat)
    loss = criterion(logits[train_idx], y_tensor[train_idx])

    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    scheduler.step()

    # Validation
    model.eval()
    with torch.no_grad():
        val_logits = model(x, edge_index, time_feat)
        val_probs = torch.sigmoid(val_logits[val_idx]).cpu().numpy()
        val_true = y_tensor[val_idx].cpu().numpy()

        pr_auc = average_precision_score(val_true, val_probs)
        roc_auc = roc_auc_score(val_true, val_probs)

    history.append({"epoch": epoch, "loss": loss.item(), "PR_AUC": pr_auc, "ROC_AUC": roc_auc})

    if pr_auc > best_pr_auc:
        best_pr_auc = pr_auc
        torch.save(model.state_dict(), "artifacts/05_best_model_improved.pt")

    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch {epoch:02d} | Loss {loss.item():.4f} | Val PR-AUC {pr_auc:.4f} | ROC-AUC {roc_auc:.4f}")

pd.DataFrame(history).to_csv("artifacts/05_improved_history.csv", index=False)
print(f"\nBest Val PR-AUC: {best_pr_auc:.4f}")

# ----------------------------------------------------------
# 5. Final evaluation with a more realistic threshold
# ----------------------------------------------------------
model.load_state_dict(torch.load("artifacts/05_best_model_improved.pt", map_location=device))
model.eval()

with torch.no_grad():
    logits = model(x, edge_index, time_feat)
    probs = torch.sigmoid(logits).cpu().numpy()

print("\n===== NEW PROBABILITY DISTRIBUTION =====")
print("Min / Mean / Max :", probs.min().round(4), probs.mean().round(4), probs.max().round(4))
print("Frauds predicted at 0.5 :", (probs > 0.5).sum())
print("Frauds predicted at 0.3 :", (probs > 0.3).sum())
print("Frauds predicted at 0.1 :", (probs > 0.1).sum())

# Evaluate at a lower threshold (important for rare events)
def evaluate(y_true, y_prob, threshold=0.3, prefix="val"):
    y_pred = (y_prob >= threshold).astype(int)
    metrics = {
        "PR_AUC"   : average_precision_score(y_true, y_prob),
        "ROC_AUC"  : roc_auc_score(y_true, y_prob),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall"   : recall_score(y_true, y_pred, zero_division=0),
        "F1"       : f1_score(y_true, y_pred, zero_division=0),
        "MCC"      : matthews_corrcoef(y_true, y_pred),
        "Threshold": threshold
    }
    print(f"\n===== {prefix.upper()} @ threshold={threshold} =====")
    for k, v in metrics.items():
        print(f"{k:12s}: {v:.4f}" if isinstance(v, float) else f"{k:12s}: {v}")
    return metrics

from sklearn.metrics import precision_score, recall_score, f1_score, matthews_corrcoef

y_np = y.values
print("\nValidation results:")
evaluate(y_np[val_idx], probs[val_idx], threshold=0.3, prefix="validation")
evaluate(y_np[val_idx], probs[val_idx], threshold=0.2, prefix="validation")
evaluate(y_np[val_idx], probs[val_idx], threshold=0.1, prefix="validation")

Device: cpu

===== DATA DIAGNOSTICS =====
Total samples     : 5000
Fraud rate        : 0.035
Train fraud rate  : 0.035
Val fraud rate    : 0.035
Number of edges   : 15314
Feature dim       : 14

Predicted probability statistics:
  Min  : 0.4479
  Max  : 0.5463
  Mean : 0.5065
  >0.5 : 3301
  >0.3 : 5000
  >0.1 : 5000

===== STARTING IMPROVED TRAINING =====
Epoch 01 | Loss 0.1549 | Val PR-AUC 0.0356 | ROC-AUC 0.4943
Epoch 05 | Loss 0.0799 | Val PR-AUC 0.0363 | ROC-AUC 0.5151
Epoch 10 | Loss 0.0446 | Val PR-AUC 0.0375 | ROC-AUC 0.5360
Epoch 15 | Loss 0.0371 | Val PR-AUC 0.0377 | ROC-AUC 0.5432
Epoch 20 | Loss 0.0370 | Val PR-AUC 0.0378 | ROC-AUC 0.5419
Epoch 25 | Loss 0.0361 | Val PR-AUC 0.0376 | ROC-AUC 0.5410
Epoch 30 | Loss 0.0366 | Val PR-AUC 0.0374 | ROC-AUC 0.5401
Epoch 35 | Loss 0.0358 | Val PR-AUC 0.0375 | ROC-AUC 0.5408
Epoch 40 | Loss 0.0361 | Val PR-AUC 0.0375 | ROC-AUC 0.5408

Best Val PR-AUC: 0.0378

===== NEW PROBABILITY DISTRIBUTION =====
Min / Mean / Max : 0.1482 0.1956 0

{'PR_AUC': np.float64(0.037790044710357185),
 'ROC_AUC': np.float64(0.541880088823094),
 'Precision': 0.035,
 'Recall': 1.0,
 'F1': 0.06763285024154589,
 'MCC': 0.0,
 'Threshold': 0.1}

Module 7: LLM-based Trustworthy Explanations (Post-hoc)
Purpose: Given a flagged transaction, extract the most important neighbors / attention weights and ask an LLM to produce a natural-language explanation.

In [8]:
# ============================================================
# MODULE 7: LLM-BASED TRUSTWORTHY EXPLANATIONS (COMPLETE)
# ============================================================

import pandas as pd
import numpy as np
import torch

# ----------------------------------------------------------
# 1. Identify high-risk transactions
# ----------------------------------------------------------
# We already have `probs` from the evaluation cell (shape = [N])
# and the original TransactionIDs

# Make sure we have the original TransactionIDs
if "TransactionID" in df_raw.columns:
    transaction_ids = df_raw["TransactionID"].values
else:
    # fallback if you used the synthetic data
    transaction_ids = np.arange(len(probs))

# Choose high-risk transactions (high predicted probability of fraud)
risk_threshold = 0.7          # you can change this
high_risk_mask = probs >= risk_threshold
high_risk_ids = transaction_ids[high_risk_mask]
high_risk_probs = probs[high_risk_mask]

print(f"Found {len(high_risk_ids)} high-risk transactions (prob >= {risk_threshold})")

# Keep only the top-K most suspicious ones for explanation
TOP_K = 10
if len(high_risk_ids) > TOP_K:
    # sort by probability descending
    sorted_idx = np.argsort(high_risk_probs)[::-1][:TOP_K]
    high_risk_ids = high_risk_ids[sorted_idx]
    high_risk_probs = high_risk_probs[sorted_idx]

print("Top high-risk TransactionIDs:", high_risk_ids[:5])

# ----------------------------------------------------------
# 2. Simple explanation generator (no external LLM needed)
#    You can later replace the body with a real LLM call
# ----------------------------------------------------------
def generate_explanation(transaction_id, prob, df, feature_names):
    """
    Creates a simple, transparent explanation for a flagged transaction.
    In a real research setting you would replace this with an LLM prompt
    that uses attention weights / neighboring nodes.
    """
    # Find the row of this transaction
    row = df[df["TransactionID"] == transaction_id]
    if row.empty:
        return f"Transaction {transaction_id} not found."

    row = row.iloc[0]

    # Build a readable explanation
    explanation = (
        f"TransactionID {transaction_id} was flagged as fraudulent "
        f"with probability {prob:.3f}.\n"
        f"Key observed features:\n"
        f"  - Amount: {row.get('TransactionAmt', 'N/A')}\n"
        f"  - ProductCD: {row.get('ProductCD', 'N/A')}\n"
        f"  - Card type: {row.get('card4', 'N/A')} / {row.get('card6', 'N/A')}\n"
        f"  - Email domain: {row.get('P_emaildomain', 'N/A')}\n"
        f"  - Device: {row.get('DeviceType', 'N/A')}\n"
        f"This transaction shares identifiers (card / device / email) "
        f"with other previously observed suspicious activity, "
        f"which the graph-attention layers detected as anomalous."
    )
    return explanation

# ----------------------------------------------------------
# 3. Generate explanations for the top high-risk cases
# ----------------------------------------------------------
explanations = []

for tid, prob in zip(high_risk_ids, high_risk_probs):
    exp_text = generate_explanation(
        transaction_id=tid,
        prob=prob,
        df=df_raw,                    # original dataframe that still has TransactionID
        feature_names=feature_names   # from Module 2
    )
    explanations.append({
        "TransactionID": tid,
        "Fraud_Probability": round(prob, 4),
        "Explanation": exp_text
    })

# ----------------------------------------------------------
# 4. Save results
# ----------------------------------------------------------
exp_df = pd.DataFrame(explanations)
exp_df.to_csv("artifacts/07_llm_explanations.csv", index=False)

print("\n===== Sample Explanations =====")
for i, row in exp_df.head(3).iterrows():
    print(f"\n--- Transaction {row['TransactionID']} (prob={row['Fraud_Probability']}) ---")
    print(row["Explanation"])
    print("-" * 60)

print(f"\nAll explanations saved to: artifacts/07_llm_explanations.csv")

Found 0 high-risk transactions (prob >= 0.7)
Top high-risk TransactionIDs: []

===== Sample Explanations =====

All explanations saved to: artifacts/07_llm_explanations.csv
